In [1]:
import os

folder_name = 'Semantic'

# Crea la cartella se non esiste
os.makedirs(folder_name, exist_ok=True)

print(f"Cartella '{folder_name}' creata o già esistente!")

Cartella 'Semantic' creata o già esistente!


In [ ]:
# =========================
# IT - CLASSIFICATORE + RISCRITTORE
# Metriche:
#  - Classificatore: Accuracy, P/R/F1 (macro+weighted), Confusion Matrix, Report, ROC-AUC, PR-AUC
#  - Riscrittore: ROUGE, BLEU (SacreBLEU), BERTScore
# In più: 1 sola inferenza finale (OUTPUT pulito, 1 frase, senza ripetizioni)
# =========================

!pip -q install -U transformers datasets evaluate rouge_score sacrebleu bert-score scikit-learn sentencepiece accelerate

import os, re, json, zipfile, shutil, random
import numpy as np
import torch
import evaluate

from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
from bert_score import score as bertscore_score

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# -------------------------
# SEED
# -------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# -------------------------
# PATHS (tuoi)
# -------------------------
# Classificatore
CLF_TRAIN_PATH = "/content/Semantic/dataset_002_it/train_classification.json"
CLF_TEST_PATH  = "/content/Semantic/dataset_002_it/test_classification.json"
CLF_ZIP        = "/content/Semantic/model_classifier_it.zip"
CLF_DIR        = "/content/models/classifier_it"

# Riscrittore
REWRITER_PATH  = "/content/Semantic/dataset_nuovo.json"
RWR_ZIP        = "/content/Semantic/model_inclusive_rewriter_it.zip"
RWR_DIR        = "/content/models/rewriter_it"

# Colonne
TEXT_COL = "text"
LABEL_COL = "label"
SRC_COL = "non_inclusiva"
TGT_COL = "inclusiva"

# Prompt 
PREFIX_MAIN     = "Riformula la frase in modo inclusivo e rispettoso, mantenendo il tema: "
PREFIX_FALLBACK = "Rewrite the sentence using inclusive language: "
PREFIX_STRICT   = "Riformula la frase in modo inclusivo e neutro, senza aggiungere dettagli non presenti: "

# Parametri generate 
GEN_KWARGS = dict(
    max_new_tokens=96,
    do_sample=False,
    num_beams=4,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2,
    length_penalty=1.0,
    early_stopping=True,
)

# -------------------------
# Utils: unzip + flatten
# -------------------------
def flatten_if_single_folder(out_dir: str):
    items = [x for x in os.listdir(out_dir) if not x.startswith(".")]
    if len(items) == 1 and os.path.isdir(os.path.join(out_dir, items[0])):
        inner = os.path.join(out_dir, items[0])
        for name in os.listdir(inner):
            shutil.move(os.path.join(inner, name), os.path.join(out_dir, name))
        shutil.rmtree(inner, ignore_errors=True)

def extract_zip(zip_path: str, out_dir: str):
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Zip non trovato: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)
    flatten_if_single_folder(out_dir)

# -------------------------
# Load riscrittore robusto (JSON/JSONL sporco)
# -------------------------
def load_rewriter_dataset_robust(path: str) -> Dataset:
    try:
        ds = load_dataset("json", data_files={"data": path})["data"]
        if SRC_COL in ds.column_names and TGT_COL in ds.column_names:
            print("Rewriter caricato con load_dataset standard.")
            return ds
        print("Caricato ma colonne diverse:", ds.column_names)
    except Exception as e:
        print("load_dataset fallito, fallback parser. Errore:", repr(e))

    records, bad_lines = [], []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        raw = f.read()

    raw_l = raw.lstrip()
    if raw_l.startswith("["):
        dec = json.JSONDecoder()
        arr, _ = dec.raw_decode(raw_l)
        records = arr if isinstance(arr, list) else []
        print(f"Letto JSON array: {len(records)} record.")
    else:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            for i, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except Exception:
                    line2 = re.sub(r",\s*$", "", line)
                    try:
                        records.append(json.loads(line2))
                    except Exception:
                        bad_lines.append(i)
        print(f"Letto JSONL: {len(records)} record validi.")
        if bad_lines:
            dump_path = "/content/bad_lines_dataset_nuovo.txt"
            with open(dump_path, "w", encoding="utf-8") as out:
                out.write("Righe corrotte (indice 1-based):\n")
                out.write("\n".join(map(str, bad_lines[:500])))
            print(f"Righe scartate: {len(bad_lines)} | dump: {dump_path}")

    cleaned = []
    for r in records:
        if not isinstance(r, dict):
            continue
        src = r.get(SRC_COL) or r.get("non_inclusive") or r.get("source") or r.get("src")
        tgt = r.get(TGT_COL) or r.get("inclusive") or r.get("target") or r.get("tgt")
        if src is None or tgt is None:
            continue
        src = str(src).strip()
        tgt = str(tgt).strip()
        if not src or not tgt:
            continue
        cleaned.append({SRC_COL: src, TGT_COL: tgt})

    print("Record finali (colonne corrette):", len(cleaned))
    return Dataset.from_list(cleaned)

# -------------------------
# Output cleaning (1 frase, no echo, no ripetizioni)
# -------------------------
SENSITIVE_WORDS = [
    "etnia", "religione", "disabilità", "handicap", "orientamento", "età", "genere",
    "razza", "nazionalità"
]

def _norm(t: str) -> str:
    t = str(t).strip()
    t = re.sub(r"\s+", " ", t)
    return t

def clean_generated(gen: str, src: str) -> str:
    s = _norm(gen)

    # togli eco del prompt (se capita)
    for p in [PREFIX_MAIN, PREFIX_FALLBACK, PREFIX_STRICT]:
        if s.lower().startswith(p.lower()):
            s = _norm(s[len(p):])

    # se inizia ripetendo esattamente l'input, taglia quella parte
    src_n = _norm(src).rstrip(".!?").lower()
    s_n = _norm(s).lower()
    if s_n.startswith(src_n) and len(s_n) > len(src_n) + 3:
        s = _norm(s[len(src):])

    # prendi solo la prima frase (webapp: 1 frase, niente papiri)
    parts = re.split(r"(?<=[.!?])\s+", s)
    parts = [x.strip() for x in parts if x.strip()]
    s = parts[0] if parts else s

    # ripetizioni brutte (parola ripetuta molte volte)
    s = re.sub(r"\b(\w+)(?:\s+\1){3,}\b", r"\1", s, flags=re.IGNORECASE)

    # rimuovi residui di virgolette/robe strane finali
    s = re.sub(r'["“”]{2,}', '"', s).strip()
    s = re.sub(r"(?:\s*[\"']\s*)+$", "", s).strip()

    # assicura punteggiatura finale
    if s and s[-1] not in ".!?":
        s += "."
    return s.strip()

def needs_regen(src: str, out: str) -> bool:
    src_n = _norm(src).rstrip(".!?").lower()
    out_n = _norm(out).rstrip(".!?").lower()

    if not out_n:
        return True
    if out_n == src_n:
        return True

    # se introduce “sensibile” non presente nell'input
    for w in SENSITIVE_WORDS:
        if (w in out_n) and (w not in src_n):
            return True

    # se è chiaramente degenerato (troppo lungo / token strani)
    if len(out) > 260:
        return True
    if "redo" in out_n or "camm" in out_n:
        return True

    return False

# -------------------------
# Device
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# 0) Estrai modelli
# -------------------------
print("\n== Extract classifier IT ==")
extract_zip(CLF_ZIP, CLF_DIR)
print("CLF_DIR files:", os.listdir(CLF_DIR)[:15], "...")

print("\n== Extract rewriter IT ==")
extract_zip(RWR_ZIP, RWR_DIR)
print("RWR_DIR files:", os.listdir(RWR_DIR)[:15], "...")

# =========================
# 1) CLASSIFICATORE — METRICHE
# =========================
print("\n====================")
print("CLASSIFICATORE (IT)")
print("====================")

clf_splits = load_dataset("json", data_files={"train": CLF_TRAIN_PATH, "test": CLF_TEST_PATH})
clf_test = clf_splits["test"]

clf_tokenizer = AutoTokenizer.from_pretrained(CLF_DIR)
clf_model = AutoModelForSequenceClassification.from_pretrained(CLF_DIR).to(device)
clf_model.eval()

# label mapping: preferisci label_map.json
label_map_path = os.path.join(CLF_DIR, "label_map.json")
label2id, id2label = None, None

if os.path.exists(label_map_path):
    with open(label_map_path, "r", encoding="utf-8") as f:
        lm = json.load(f)
    if isinstance(lm, dict) and lm and all(isinstance(v, int) for v in lm.values()):
        label2id = lm
        id2label = {int(v): k for k, v in label2id.items()}
    elif isinstance(lm, dict) and lm and all(str(k).isdigit() for k in lm.keys()):
        id2label = {int(k): v for k, v in lm.items()}
        label2id = {v: int(k) for k, v in id2label.items()}

# fallback: config
if label2id is None and getattr(clf_model.config, "label2id", None):
    if isinstance(clf_model.config.label2id, dict) and len(clf_model.config.label2id) > 0:
        try:
            label2id = {k: int(v) for k, v in clf_model.config.label2id.items()}
            id2label = {v: k for k, v in label2id.items()}
        except Exception:
            label2id = None

# fallback: dataset
if label2id is None:
    uniq = sorted(list(set([str(x) for x in clf_test[LABEL_COL]])))
    label2id = {lab: i for i, lab in enumerate(uniq)}
    id2label = {i: lab for lab, i in label2id.items()}

print("Label mapping:", label2id)

# positivo per AUC: preferisci "non_inclusiva"
pos_name = None
for cand in ["non_inclusiva", "not_inclusive", "non-inclusiva", "not inclusive"]:
    for k in label2id.keys():
        if str(k).lower() == cand:
            pos_name = k
            break
    if pos_name is not None:
        break
if pos_name is None:
    keys = list(label2id.keys())
    pos_name = keys[1] if len(keys) > 1 else keys[0]
pos_id = int(label2id[pos_name])

def clf_collate(batch):
    texts = [str(x[TEXT_COL]) for x in batch]
    labels_raw = [str(x[LABEL_COL]) for x in batch]
    labels = [int(label2id[l]) for l in labels_raw]
    enc = clf_tokenizer(texts, truncation=True, padding=True, max_length=256, return_tensors="pt")
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

clf_loader = DataLoader(clf_test, batch_size=32, shuffle=False, collate_fn=clf_collate)

all_preds, all_labels, all_scores = [], [], []
with torch.no_grad():
    for batch in clf_loader:
        labels = batch.pop("labels").cpu().numpy()
        batch = {k: v.to(device) for k, v in batch.items()}
        out = clf_model(**batch)
        probs = torch.softmax(out.logits, dim=-1).detach().cpu().numpy()
        preds = np.argmax(probs, axis=-1)
        scores = probs[:, pos_id] if probs.shape[1] > 1 else probs[:, 0]
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
        all_scores.extend(scores.tolist())

acc = accuracy_score(all_labels, all_preds)
p_w, r_w, f1_w, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
p_m, r_m, f1_m, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)

print("Accuracy:", acc)
print("Weighted  P/R/F1:", p_w, r_w, f1_w)
print("Macro     P/R/F1:", p_m, r_m, f1_m)
print("\nConfusion matrix:\n", confusion_matrix(all_labels, all_preds))

target_names = [id2label[i] for i in range(len(id2label))]
print("\nClassification report:\n", classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))

if len(set(all_labels)) == 2:
    y_true_bin = np.array([1 if y == pos_id else 0 for y in all_labels], dtype=np.int32)
    y_score = np.array(all_scores, dtype=np.float32)
    print("ROC-AUC:", roc_auc_score(y_true_bin, y_score))
    print("PR-AUC :", average_precision_score(y_true_bin, y_score))
else:
    print("ROC-AUC / PR-AUC: saltate (task non binario)")

# =========================
# 2) RISCRITTORE — METRICHE (ROUGE / BLEU / BERTScore)
# =========================
print("\n====================")
print("RISCRITTORE (IT)")
print("====================")

# Dataset (lasciato così com'è, con parser robusto se serve)
rew_all = load_rewriter_dataset_robust(REWRITER_PATH)
tmp = rew_all.train_test_split(test_size=0.1, seed=SEED)
rew_test = tmp["test"]

rwr_tokenizer = AutoTokenizer.from_pretrained(RWR_DIR)
rwr_model = AutoModelForSeq2SeqLM.from_pretrained(RWR_DIR).to(device)
rwr_model.eval()

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

src_texts = [str(x[SRC_COL]) for x in rew_test]
ref_texts = [str(x[TGT_COL]) for x in rew_test]

@torch.inference_mode()
def generate_with_prefix(texts, prefix: str, batch_size=16):
    outs = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        inputs = [prefix + _norm(t) for t in chunk]
        enc = rwr_tokenizer(
            inputs,
            truncation=True,
            max_length=256,
            padding=True,
            return_tensors="pt"
        ).to(device)

        gen = rwr_model.generate(**enc, **GEN_KWARGS)
        decoded = rwr_tokenizer.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        outs.extend([d.strip() for d in decoded])
    return outs

def multi_pass_generate(src_list, batch_size=16):
    # Pass 1: app.py prefix
    raw = generate_with_prefix(src_list, PREFIX_MAIN, batch_size=batch_size)
    clean = [clean_generated(r, s) for r, s in zip(raw, src_list)]
    bad = [i for i,(s,o) in enumerate(zip(src_list, clean)) if needs_regen(s,o)]

    # Pass 2: english prefix (service_it.py) solo sui bad
    if bad:
        raw2 = generate_with_prefix([src_list[i] for i in bad], PREFIX_FALLBACK, batch_size=batch_size)
        clean2 = [clean_generated(r, src_list[i]) for r, i in zip(raw2, bad)]
        for j, idx in enumerate(bad):
            raw[idx] = raw2[j]
            clean[idx] = clean2[j]
        bad = [i for i,(s,o) in enumerate(zip(src_list, clean)) if needs_regen(s,o)]

    # Pass 3: strict italian fallback solo sui bad residui
    if bad:
        raw3 = generate_with_prefix([src_list[i] for i in bad], PREFIX_STRICT, batch_size=batch_size)
        clean3 = [clean_generated(r, src_list[i]) for r, i in zip(raw3, bad)]
        for j, idx in enumerate(bad):
            raw[idx] = raw3[j]
            clean[idx] = clean3[j]

    return clean

pred_clean_final = multi_pass_generate(src_texts, batch_size=16)

# Metriche su output "finale" (pulito)
rouge_res = rouge.compute(predictions=pred_clean_final, references=ref_texts, use_stemmer=False)
rouge_res = {k: float(v) * 100 for k, v in rouge_res.items()}
print("ROUGE (IT) [final]:", rouge_res)

bleu_res = bleu.compute(predictions=pred_clean_final, references=[[r] for r in ref_texts])
print("BLEU (SacreBLEU, IT) [final]:", bleu_res)

bs_model = "xlm-roberta-base"
P, R, F1 = bertscore_score(
    cands=pred_clean_final,
    refs=ref_texts,
    lang="it",
    model_type=bs_model,
    device=device,
    batch_size=8,   # se OOM: 4
    verbose=False
)
print("BERTScore (IT) model:", bs_model)
print("BERTScore mean P/R/F1 [final]:",
      float(P.mean().item()),
      float(R.mean().item()),
      float(F1.mean().item()))

# =========================
# 3) UNA SOLA INFERENZA (solo output pulito, 1 frase)
# =========================
TEST_SENTENCE = "Le donne non sanno guidare."
single_out = multi_pass_generate([TEST_SENTENCE], batch_size=1)[0]
print("\nOUTPUT_RISCRITTORE:", single_out)


Device: cuda
GPU: Tesla T4

== Extract classifier IT ==
✅ CLF_DIR files: ['special_tokens_map.json', 'model.safetensors', 'label_map.json', 'tokenizer.json', 'training_args.bin', 'vocab.txt', 'config.json', 'tokenizer_config.json'] ...

== Extract rewriter IT ==
✅ RWR_DIR files: ['special_tokens_map.json', 'generation_config.json', 'model.safetensors', 'tokenizer.json', 'training_args.bin', 'trainer_state.json', 'config.json', 'tokenizer_config.json'] ...

CLASSIFICATORE (IT)
Label mapping: {'inclusiva': 0, 'non_inclusiva': 1}
Accuracy: 0.9924834636199639
Weighted  P/R/F1: 0.9924850663156829 0.9924834636199639 0.9924834575046868
Macro     P/R/F1: 0.9924850663156829 0.992483463619964 0.9924834575046868

Confusion matrix:
 [[3304   22]
 [  28 3298]]

Classification report:
                precision    recall  f1-score   support

    inclusiva       0.99      0.99      0.99      3326
non_inclusiva       0.99      0.99      0.99      3326

     accuracy                           0.99      

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cm = np.array([[3304, 22],
               [  28, 3298]])

labels = ["inclusive", "non-inclusive"]

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.xticks([0,1], labels, rotation=15)
plt.yticks([0,1], labels)

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center')

plt.title("Confusion Matrix (Italian Classifier)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig("confusion_matrix_it.png", dpi=200)
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# y_true: 0/1, y_score: prob for class 1
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

prec, rec, _ = precision_recall_curve(y_true, y_score)
pr_auc = auc(rec, prec)

plt.figure()
plt.plot(fpr, tpr)
plt.title(f"ROC Curve (AUC={roc_auc:.4f})")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.tight_layout()
plt.savefig("roc_it.png", dpi=200)
plt.show()

plt.figure()
plt.plot(rec, prec)
plt.title(f"PR Curve (AUC={pr_auc:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.tight_layout()
plt.savefig("pr_it.png", dpi=200)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "BERTScore F1"]
values  = [23.2674, 9.1915, 21.0815, 5.9773, 0.8859]

plt.figure(figsize=(7,4))
plt.bar(metrics, values)
plt.title("Italian Rewriter — Automatic Metrics")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("rewriter_metrics_it.png", dpi=200)
plt.show()
